# Structural Hypertransmission Pipeline Validation

This notebook applies the structural-exclusion and hypertransmission pipeline to
a fixed set of fast- and slow-progressor B-scans.

The purpose is to evaluate whether the configuration developed on one scan
generalizes across subjects without additional scan-specific tuning.

For every scan, the same sequence is applied:

1. Flatten to Bruch's membrane.
2. Crop a 150-pixel sub-BM region.
3. Apply whole-ROI z-score normalization.
4. Apply anisotropic Gaussian denoising.
5. Estimate simple gradient-based verticality.
6. Exclude pixels that are both vertically oriented and above the scan's 80th
   gradient-magnitude percentile.
7. Calculate structurally cleaned column median and 90th-percentile intensity.
8. Construct a conservative hypertransmission mask using robust intensity
   thresholds.
9. Refine hypertransmission using the 60th percentile of depth continuity.
10. Refine candidate barcoding using the 70th percentile of strong vertical
    pixel fraction.

The parameter rules are fixed across scans. Quantile-based thresholds are
recomputed within each scan because they are defined relative to that scan's
signal distribution.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the project root."
        )

    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

from src.detector.data import (
    build_grouped_volume_registry,
    load_e2e_volume,
)

from src.detector.preprocessing import (
    preprocess_bscan,
)

from src.detector.features import (
    compute_column_intensity_statistics,
    compute_simple_verticality_map,
    create_structural_mask,
)

print("Project root:", PROJECT_ROOT)

## Validation Cohort

An explicit B-scan index is selected for each subject. Scan selection remains
fixed throughout validation.

The progression group is used only to organize the visualizations. It is not
used by the segmentation pipeline.

In [ ]:
E2E_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "heyex"
    / "meta"
)

PROGRESSION_GROUPS = {
    "fast": [
        8,
        9,
        12,
        41,
        49,
    ],
    "slow": [
        17,
        23,
        35,
        36,
        47,
    ],
}

registry = build_grouped_volume_registry(
    e2e_directory=E2E_DIRECTORY,
    progression_groups=PROGRESSION_GROUPS,
)

print(
    "Registry entries:",
    len(registry),
)

In [ ]:
SELECTED_SCAN_CONFIG = {
    "fast": {
        8: 54,
        9: 85,
        12: 38,
        41: 59,
        49: 33,
    },
    "slow": {
        17: 55,
        23: 43,
        35: 38,
        36: 59,
        47: 31,
    },
}

## Frozen Pipeline Configuration

The following configuration was selected before examining the validation
results. These values must remain unchanged while the selected scans are
processed.

A quantile rule such as `gradient_quantile = 0.80` means that the numerical
threshold is recalculated from each scan, but the same 80th-percentile rule is
used for every subject.

In [ ]:
PREPROCESSING_CONFIG = {
    "layer_name": "BM",

    # Flattening
    "reference_row": None,
    "flatten_fill_value": 0.0,

    # Sub-layer crop
    "depth_below_layer": 150,
    "include_boundary": True,
    "require_full_depth": False,
    "crop_fill_value": 0.0,

    # Whole-ROI normalization
    "normalization_method": "zscore",
    "lower_percentile": 1.0,
    "upper_percentile": 99.0,

    # Anisotropic Gaussian denoising
    "denoise_method": "gaussian",
    "gaussian_sigma": (
        1.0,
        0.5,
    ),
}

In [ ]:
PIPELINE_CONFIG = {
    # Simple verticality
    "verticality_smoothing_sigma": 1.0,
    "verticality_threshold": 0.60,

    # Strong-gradient structural exclusion
    "gradient_quantile": 0.80,
    "minimum_component_size": 0,

    # Column intensity statistics
    "column_upper_quantile": 0.90,
    "minimum_valid_pixels": 5,

    # Horizontal signal smoothing
    "signal_smoothing_sigma": 2.0,

    # Robust intensity thresholds
    "median_iqr_multiplier": 1.0,
    "q90_iqr_multiplier": 0.5,

    # Depth continuity
    "continuity_window_width": 15,
    "continuity_depth_lag": 4,
    "continuity_minimum_row_standard_deviation": 1e-6,
    "continuity_quantile": 0.60,

    # Final vertical refinement
    "vertical_fraction_quantile": 0.70,

    # Mask cleanup
    "minimum_positive_run": 5,
    "maximum_negative_gap": 2,

    # Edge handling
    "edge_margin": 10,
}

In [ ]:
def smooth_finite_signal(
    values: np.ndarray,
    *,
    sigma: float,
) -> np.ndarray:
    """
    Smooth a one-dimensional signal while interpolating non-finite values.
    """
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    if values.ndim != 1:
        raise ValueError(
            "values must be one-dimensional."
        )

    finite = np.isfinite(values)

    if not finite.any():
        raise ValueError(
            "The signal contains no finite values."
        )

    filled = values.copy()

    if not finite.all():
        positions = np.arange(
            values.size
        )

        filled[
            ~finite
        ] = np.interp(
            positions[~finite],
            positions[finite],
            values[finite],
        )

    return gaussian_filter1d(
        filled,
        sigma=float(sigma),
        mode="nearest",
    ).astype(np.float32)

In [ ]:
def find_boolean_runs(
    mask: np.ndarray,
    target_value: bool = True,
) -> list[tuple[int, int]]:
    """
    Return inclusive start and end positions for Boolean runs.
    """
    mask = np.asarray(
        mask,
        dtype=bool,
    )

    if mask.ndim != 1:
        raise ValueError(
            "mask must be one-dimensional."
        )

    runs: list[
        tuple[int, int]
    ] = []

    run_start = None

    for position, value in enumerate(mask):
        if bool(value) == bool(target_value):
            if run_start is None:
                run_start = position

        elif run_start is not None:
            runs.append(
                (
                    run_start,
                    position - 1,
                )
            )

            run_start = None

    if run_start is not None:
        runs.append(
            (
                run_start,
                mask.size - 1,
            )
        )

    return runs


def clean_column_mask(
    mask: np.ndarray,
    *,
    minimum_positive_run: int,
    maximum_negative_gap: int,
) -> np.ndarray:
    """
    Fill short internal gaps and remove short positive runs.
    """
    cleaned = np.asarray(
        mask,
        dtype=bool,
    ).copy()

    if minimum_positive_run < 1:
        raise ValueError(
            "minimum_positive_run must be positive."
        )

    if maximum_negative_gap < 0:
        raise ValueError(
            "maximum_negative_gap must be nonnegative."
        )

    for start, end in find_boolean_runs(
        cleaned,
        target_value=False,
    ):
        gap_length = end - start + 1

        touches_edge = (
            start == 0
            or end == cleaned.size - 1
        )

        if (
            not touches_edge
            and gap_length
            <= maximum_negative_gap
        ):
            cleaned[
                start:
                end + 1
            ] = True

    for start, end in find_boolean_runs(
        cleaned,
        target_value=True,
    ):
        run_length = end - start + 1

        if run_length < minimum_positive_run:
            cleaned[
                start:
                end + 1
            ] = False

    return cleaned

In [ ]:
def compute_local_depth_continuity(
    image: np.ndarray,
    *,
    window_width: int = 15,
    depth_lag: int = 4,
    minimum_row_standard_deviation: float = 1e-6,
) -> np.ndarray:
    """
    Calculate one local depth-continuity value per image column.
    """
    image = np.asarray(
        image,
        dtype=np.float32,
    )

    if image.ndim != 2:
        raise ValueError(
            "image must be two-dimensional."
        )

    if (
        window_width <= 1
        or window_width % 2 == 0
    ):
        raise ValueError(
            "window_width must be an odd integer greater than one."
        )

    if not (
        1
        <= depth_lag
        < image.shape[0]
    ):
        raise ValueError(
            "depth_lag must be between 1 and image depth - 1."
        )

    radius = window_width // 2

    padded_image = np.pad(
        image,
        (
            (0, 0),
            (radius, radius),
        ),
        mode="reflect",
    )

    continuity = np.full(
        image.shape[1],
        np.nan,
        dtype=np.float32,
    )

    for horizontal_position in range(
        image.shape[1]
    ):
        local_window = padded_image[
            :,
            horizontal_position:
            horizontal_position
            + window_width,
        ]

        correlations: list[float] = []

        for depth_position in range(
            image.shape[0]
            - depth_lag
        ):
            first_row = local_window[
                depth_position
            ]

            second_row = local_window[
                depth_position
                + depth_lag
            ]

            if (
                np.std(first_row)
                < minimum_row_standard_deviation
                or np.std(second_row)
                < minimum_row_standard_deviation
            ):
                continue

            correlation = np.corrcoef(
                first_row,
                second_row,
            )[0, 1]

            if np.isfinite(correlation):
                correlations.append(
                    float(correlation)
                )

        if correlations:
            continuity[
                horizontal_position
            ] = float(
                np.median(
                    correlations
                )
            )

    return continuity

In [ ]:
@dataclass
class StructuralHyperTDValidationResult:
    """
    Pipeline output for one selected B-scan.
    """

    subject_id: int
    progression_group: str
    bscan_index: int

    processed: Any

    simple_verticality_map: np.ndarray
    gradient_magnitude: np.ndarray
    structural_mask: np.ndarray

    column_median_smoothed: np.ndarray
    column_q90_smoothed: np.ndarray

    continuity_smoothed: np.ndarray
    strong_vertical_fraction_smoothed: np.ndarray

    intensity_mask: np.ndarray
    continuity_mask: np.ndarray
    barcode_mask: np.ndarray

    thresholds: dict[str, float]

In [ ]:
def run_structural_hypertd_pipeline(
    *,
    volume,
    subject_id: int,
    progression_group: str,
    bscan_index: int,
    preprocessing_config: dict[str, Any],
    pipeline_config: dict[str, Any],
) -> StructuralHyperTDValidationResult:
    """
    Apply the frozen structural-hypertransmission pipeline to one B-scan.
    """
    processed = preprocess_bscan(
        volume=volume,
        bscan_index=bscan_index,
        **preprocessing_config,
    )

    image = np.asarray(
        processed.denoised_scan,
        dtype=np.float32,
    )

    image_height, image_width = (
        image.shape
    )

    edge_margin = int(
        pipeline_config[
            "edge_margin"
        ]
    )

    # --------------------------------------------------------------
    # 1. Simple verticality and gradient magnitude
    # --------------------------------------------------------------

    (
        simple_verticality_map,
        verticality_diagnostics,
    ) = compute_simple_verticality_map(
        image=image,
        smoothing_sigma=float(
            pipeline_config[
                "verticality_smoothing_sigma"
            ]
        ),
    )

    gradient_magnitude = np.hypot(
        verticality_diagnostics[
            "gradient_x"
        ],
        verticality_diagnostics[
            "gradient_z"
        ],
    ).astype(np.float32)

    interior_pixel_mask = np.ones(
        image.shape,
        dtype=bool,
    )

    interior_pixel_mask[
        :,
        :edge_margin,
    ] = False

    interior_pixel_mask[
        :,
        image_width - edge_margin:,
    ] = False

    gradient_reference = (
        gradient_magnitude[
            interior_pixel_mask
        ]
    )

    gradient_threshold = float(
        np.quantile(
            gradient_reference,
            pipeline_config[
                "gradient_quantile"
            ],
        )
    )

    (
        structural_mask,
        _structural_metadata,
    ) = create_structural_mask(
        verticality_map=(
            simple_verticality_map
        ),
        gradient_magnitude=(
            gradient_magnitude
        ),
        verticality_threshold=float(
            pipeline_config[
                "verticality_threshold"
            ]
        ),
        minimum_gradient_magnitude=(
            gradient_threshold
        ),
        minimum_component_size=int(
            pipeline_config[
                "minimum_component_size"
            ]
        ),
    )

    # --------------------------------------------------------------
    # 2. Structurally cleaned column intensity
    # --------------------------------------------------------------

    (
        column_statistics,
        _column_metadata,
    ) = compute_column_intensity_statistics(
        image=image,
        exclusion_mask=structural_mask,
        upper_quantile=float(
            pipeline_config[
                "column_upper_quantile"
            ]
        ),
        minimum_valid_pixels=int(
            pipeline_config[
                "minimum_valid_pixels"
            ]
        ),
    )

    signal_sigma = float(
        pipeline_config[
            "signal_smoothing_sigma"
        ]
    )

    column_median_smoothed = (
        smooth_finite_signal(
            column_statistics[
                "median"
            ],
            sigma=signal_sigma,
        )
    )

    column_q90_smoothed = (
        smooth_finite_signal(
            column_statistics[
                "upper_quantile"
            ],
            sigma=signal_sigma,
        )
    )

    reference_columns = np.ones(
        image_width,
        dtype=bool,
    )

    reference_columns[
        :edge_margin
    ] = False

    reference_columns[
        -edge_margin:
    ] = False

    median_reference = (
        column_median_smoothed[
            reference_columns
        ]
    )

    q90_reference = (
        column_q90_smoothed[
            reference_columns
        ]
    )

    median_reference_median = float(
        np.median(
            median_reference
        )
    )

    median_reference_iqr = float(
        np.quantile(
            median_reference,
            0.75,
        )
        - np.quantile(
            median_reference,
            0.25,
        )
    )

    q90_reference_median = float(
        np.median(
            q90_reference
        )
    )

    q90_reference_iqr = float(
        np.quantile(
            q90_reference,
            0.75,
        )
        - np.quantile(
            q90_reference,
            0.25,
        )
    )

    median_threshold = (
        median_reference_median
        + float(
            pipeline_config[
                "median_iqr_multiplier"
            ]
        )
        * median_reference_iqr
    )

    q90_threshold = (
        q90_reference_median
        + float(
            pipeline_config[
                "q90_iqr_multiplier"
            ]
        )
        * q90_reference_iqr
    )

    intensity_mask = (
        (
            column_median_smoothed
            > median_threshold
        )
        & (
            column_q90_smoothed
            > q90_threshold
        )
    )

    intensity_mask[
        :edge_margin
    ] = False

    intensity_mask[
        -edge_margin:
    ] = False

    intensity_mask = clean_column_mask(
        intensity_mask,
        minimum_positive_run=int(
            pipeline_config[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=int(
            pipeline_config[
                "maximum_negative_gap"
            ]
        ),
    )

    # --------------------------------------------------------------
    # 3. Depth-continuity refinement
    # --------------------------------------------------------------

    continuity_signal = (
        compute_local_depth_continuity(
            image,
            window_width=int(
                pipeline_config[
                    "continuity_window_width"
                ]
            ),
            depth_lag=int(
                pipeline_config[
                    "continuity_depth_lag"
                ]
            ),
            minimum_row_standard_deviation=float(
                pipeline_config[
                    "continuity_minimum_row_standard_deviation"
                ]
            ),
        )
    )

    continuity_smoothed = (
        smooth_finite_signal(
            continuity_signal,
            sigma=signal_sigma,
        )
    )

    continuity_reference = (
        continuity_smoothed[
            reference_columns
        ]
    )

    continuity_threshold = float(
        np.quantile(
            continuity_reference,
            pipeline_config[
                "continuity_quantile"
            ],
        )
    )

    continuity_mask = (
        intensity_mask
        & (
            continuity_smoothed
            > continuity_threshold
        )
    )

    continuity_mask = clean_column_mask(
        continuity_mask,
        minimum_positive_run=int(
            pipeline_config[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=int(
            pipeline_config[
                "maximum_negative_gap"
            ]
        ),
    )

    # --------------------------------------------------------------
    # 4. Strong vertical-fraction refinement
    # --------------------------------------------------------------

    strong_vertical_fraction = np.mean(
        structural_mask,
        axis=0,
    ).astype(np.float32)

    strong_vertical_fraction_smoothed = (
        smooth_finite_signal(
            strong_vertical_fraction,
            sigma=signal_sigma,
        )
    )

    vertical_fraction_reference = (
        strong_vertical_fraction_smoothed[
            reference_columns
        ]
    )

    vertical_fraction_threshold = float(
        np.quantile(
            vertical_fraction_reference,
            pipeline_config[
                "vertical_fraction_quantile"
            ],
        )
    )

    barcode_mask = (
        continuity_mask
        & (
            strong_vertical_fraction_smoothed
            > vertical_fraction_threshold
        )
    )

    barcode_mask[
        :edge_margin
    ] = False

    barcode_mask[
        -edge_margin:
    ] = False

    barcode_mask = clean_column_mask(
        barcode_mask,
        minimum_positive_run=int(
            pipeline_config[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=int(
            pipeline_config[
                "maximum_negative_gap"
            ]
        ),
    )

    thresholds = {
        "gradient_threshold": (
            gradient_threshold
        ),
        "median_threshold": (
            median_threshold
        ),
        "q90_threshold": (
            q90_threshold
        ),
        "continuity_threshold": (
            continuity_threshold
        ),
        "vertical_fraction_threshold": (
            vertical_fraction_threshold
        ),
    }

    return StructuralHyperTDValidationResult(
        subject_id=int(
            subject_id
        ),
        progression_group=str(
            progression_group
        ),
        bscan_index=int(
            bscan_index
        ),
        processed=processed,
        simple_verticality_map=(
            simple_verticality_map
        ),
        gradient_magnitude=(
            gradient_magnitude
        ),
        structural_mask=(
            structural_mask
        ),
        column_median_smoothed=(
            column_median_smoothed
        ),
        column_q90_smoothed=(
            column_q90_smoothed
        ),
        continuity_smoothed=(
            continuity_smoothed
        ),
        strong_vertical_fraction_smoothed=(
            strong_vertical_fraction_smoothed
        ),
        intensity_mask=(
            intensity_mask
        ),
        continuity_mask=(
            continuity_mask
        ),
        barcode_mask=(
            barcode_mask
        ),
        thresholds=thresholds,
    )

## Run Frozen Pipeline

All configured scans are processed using the same pipeline configuration.

The results are stored in memory for visual inspection. No parameters are
changed between subjects.

In [ ]:
validation_results: dict[
    tuple[str, int],
    StructuralHyperTDValidationResult,
] = {}

registry_lookup = {
    (
        str(record.progression_group).lower(),
        int(record.subject_id),
    ): record
    for record in registry
}

for progression_group, subject_scans in (
    SELECTED_SCAN_CONFIG.items()
):
    for subject_id, bscan_index in (
        subject_scans.items()
    ):
        lookup_key = (
            progression_group.lower(),
            int(subject_id),
        )

        if lookup_key not in registry_lookup:
            raise KeyError(
                f"Could not find {progression_group} "
                f"subject {subject_id} in the registry."
            )

        record = registry_lookup[
            lookup_key
        ]

        volume = load_e2e_volume(
            record.e2e_path
        )

        if not (
            0
            <= bscan_index
            < len(volume)
        ):
            raise IndexError(
                f"B-scan {bscan_index} is outside the valid range "
                f"0 to {len(volume) - 1} for subject {subject_id}."
            )

        result = (
            run_structural_hypertd_pipeline(
                volume=volume,
                subject_id=subject_id,
                progression_group=(
                    progression_group
                ),
                bscan_index=bscan_index,
                preprocessing_config=(
                    PREPROCESSING_CONFIG
                ),
                pipeline_config=(
                    PIPELINE_CONFIG
                ),
            )
        )

        validation_results[
            lookup_key
        ] = result

        print(
            f"Completed {progression_group} "
            f"subject {subject_id}, "
            f"B-scan {bscan_index}"
        )

## Group Validation Visualizations

Each row displays one selected scan. The shaded regions are the final candidate
barcoding intervals after intensity, depth-continuity, and vertical-fraction
refinement.

The progression groups are displayed separately only for organization.

In [ ]:
def plot_validation_group(
    validation_results: dict[
        tuple[str, int],
        StructuralHyperTDValidationResult,
    ],
    progression_group: str,
    *,
    overlay_color: str,
    overlay_alpha: float = 0.30,
    figure_size: tuple[float, float] = (
        16,
        15,
    ),
):
    """
    Plot final candidate barcode intervals for one progression group.
    """
    resolved_group = str(
        progression_group
    ).strip().lower()

    matching_results = [
        result
        for (
            group_name,
            _subject_id,
        ), result in validation_results.items()
        if group_name == resolved_group
    ]

    if not matching_results:
        raise KeyError(
            f"No validation results were found for group "
            f"'{progression_group}'."
        )

    matching_results.sort(
        key=lambda result: result.subject_id
    )

    figure, axes = plt.subplots(
        nrows=len(
            matching_results
        ),
        ncols=1,
        figsize=figure_size,
        squeeze=False,
    )

    flat_axes = axes.ravel()

    for axis, result in zip(
        flat_axes,
        matching_results,
    ):
        image = np.asarray(
            result.processed.denoised_scan,
            dtype=np.float32,
        )

        image_height, image_width = (
            image.shape
        )

        axis.imshow(
            image,
            cmap="gray",
            aspect="auto",
            extent=(
                -0.5,
                image_width - 0.5,
                image_height - 0.5,
                -0.5,
            ),
        )

        for start, end in find_boolean_runs(
            result.barcode_mask,
            target_value=True,
        ):
            axis.axvspan(
                start,
                end,
                color=overlay_color,
                alpha=overlay_alpha,
                linewidth=0,
            )

        axis.set_title(
            f"Subject {result.subject_id} | "
            f"{resolved_group.title()} progressor | "
            f"B-scan {result.bscan_index}"
        )

        axis.set_ylabel(
            "Depth below BM"
        )

        axis.set_xlim(
            -0.5,
            image_width - 0.5,
        )

        axis.set_ylim(
            image_height - 0.5,
            -0.5,
        )

    flat_axes[-1].set_xlabel(
        "Horizontal position"
    )

    figure.suptitle(
        f"{resolved_group.title()} progressors: "
        "provisional candidate barcoding",
        fontsize=16,
        y=1.01,
    )

    figure.tight_layout()

    return figure, axes

In [ ]:
fast_figure, fast_axes = (
    plot_validation_group(
        validation_results=(
            validation_results
        ),
        progression_group="fast",
        overlay_color="tab:red",
        overlay_alpha=0.30,
        figure_size=(16, 15),
    )
)

plt.show()

In [ ]:
slow_figure, slow_axes = (
    plot_validation_group(
        validation_results=(
            validation_results
        ),
        progression_group="slow",
        overlay_color="tab:green",
        overlay_alpha=0.30,
        figure_size=(16, 15),
    )
)

plt.show()

## Individual Failure-Mode Inspection

A selected validation case can be inspected stage by stage without rerunning the
pipeline.

This view is used only after the frozen group-level results have been generated.
It helps determine whether a missed or excessive detection originates from:

- the intensity criterion;
- depth continuity;
- or final vertical-fraction refinement.

In [ ]:
INSPECT_GROUP = "fast"
INSPECT_SUBJECT = 8

In [ ]:
inspection_key = (
    INSPECT_GROUP.lower(),
    int(INSPECT_SUBJECT),
)

if inspection_key not in validation_results:
    raise KeyError(
        f"No validation result exists for "
        f"{inspection_key}."
    )

inspection_result = (
    validation_results[
        inspection_key
    ]
)

inspection_image = np.asarray(
    inspection_result
    .processed
    .denoised_scan,
    dtype=np.float32,
)

image_height, image_width = (
    inspection_image.shape
)

inspection_stages = (
    (
        "Denoised sub-BM scan",
        None,
        None,
    ),
    (
        "Conservative intensity candidate",
        inspection_result.intensity_mask,
        "tab:red",
    ),
    (
        "Depth-continuity refinement",
        inspection_result.continuity_mask,
        "tab:blue",
    ),
    (
        "Final vertical-fraction refinement",
        inspection_result.barcode_mask,
        "tab:purple",
    ),
)

fig, axes = plt.subplots(
    5,
    1,
    figsize=(15, 15),
    sharex=True,
)

for axis, (
    title,
    mask,
    color,
) in zip(
    axes[:4],
    inspection_stages,
):
    axis.imshow(
        inspection_image,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            image_height - 0.5,
            -0.5,
        ),
    )

    if mask is not None:
        for start, end in (
            find_boolean_runs(
                mask,
                target_value=True,
            )
        ):
            axis.axvspan(
                start,
                end,
                color=color,
                alpha=0.30,
                linewidth=0,
            )

    axis.set_title(
        title
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[4].plot(
    inspection_result.continuity_smoothed,
    label="Depth continuity",
)

axes[4].axhline(
    inspection_result.thresholds[
        "continuity_threshold"
    ],
    linestyle="--",
    label="Continuity threshold",
)

axes[4].plot(
    inspection_result
    .strong_vertical_fraction_smoothed,
    label="Strong vertical fraction",
)

axes[4].axhline(
    inspection_result.thresholds[
        "vertical_fraction_threshold"
    ],
    linestyle=":",
    label="Vertical-fraction threshold",
)

for start, end in find_boolean_runs(
    inspection_result.barcode_mask,
    target_value=True,
):
    axes[4].axvspan(
        start,
        end,
        color="tab:purple",
        alpha=0.15,
    )

axes[4].set_title(
    "Final refinement signals"
)

axes[4].set_xlabel(
    "Horizontal position"
)

axes[4].set_ylabel(
    "Signal value"
)

axes[4].legend(
    ncols=2
)

axes[4].grid(
    alpha=0.25
)

fig.suptitle(
    f"Subject {inspection_result.subject_id} | "
    f"{inspection_result.progression_group.title()} progressor | "
    f"B-scan {inspection_result.bscan_index}",
    fontsize=15,
    y=1.01,
)

plt.tight_layout()
plt.show()